In [18]:
import kagglehub
import numpy as np

# Download latest version
path = kagglehub.dataset_download("maseratiurm/multi-class-prediction-of-obesity-risk")

print("Path to dataset files:", path)

100%|██████████| 917k/917k [00:00<00:00, 1.53MB/s]

Extracting files...
Path to dataset files: /home/franio/.cache/kagglehub/datasets/maseratiurm/multi-class-prediction-of-obesity-risk/versions/1


In [25]:
import pandas as pd
import os

data_path = os.path.join(path, "train.csv", "train.csv")
data = pd.read_csv(data_path)

X, y = data.drop(["NObeyesdad", "id"], axis=1), data["NObeyesdad"]

X

,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS
0,Male,24.443011,1.699998,81.669950,yes,yes,2.000000,2.983297,Sometimes,no,2.763573,no,0.000000,0.976473,Sometimes,Public_Transportation
1,Female,18.000000,1.560000,57.000000,yes,yes,2.000000,3.000000,Frequently,no,2.000000,no,1.000000,1.000000,no,Automobile
2,Female,18.000000,1.711460,50.165754,yes,yes,1.880534,1.411685,Sometimes,no,1.910378,no,0.866045,1.673584,no,Public_Transportation
3,Female,20.952737,1.710730,131.274851,yes,yes,3.000000,3.000000,Sometimes,no,1.674061,no,1.467863,0.780199,Sometimes,Public_Transportation
4,Male,31.641081,1.914186,93.798055,yes,yes,2.679664,1.971472,Sometimes,no,1.979848,no,1.967973,0.931721,Sometimes,Public_Transportation
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20753,Male,25.137087,1.766626,114.187096,yes,yes,2.919584,3.000000,Sometimes,no,2.151809,no,1.330519,0.196680,Sometimes,Public_Transportation
20754,Male,18.000000,1.710000,50.000000,no,yes,3.000000,4.000000,Frequently,no,1.000000,no,2.000000,1.000000,Sometimes,Public_Transportation
20755,Male,20.101026,1.819557,105.580491,yes,yes,2.407817,3.000000,Sometimes,no,2.000000,no,1.158040,1.198439,no,Public_Transportation
20756,Male,33.852953,1.700000,83.520113,yes,yes,2.671238,1.971472,Sometimes,no,2.144838,no,0.000000,0.973834,no,Automobile


In [31]:
print(data.dtypes.sort_values())

id                                  int64
TUE                               float64
FAF                               float64
CH2O                              float64
FCVC                              float64
NCP                               float64
Weight                            float64
Height                            float64
Age                               float64
FAVC                               object
MTRANS                             object
CAEC                               object
SMOKE                              object
SCC                                object
Gender                             object
CALC                               object
family_history_with_overweight     object
NObeyesdad                         object
dtype: object


## Check for Nans and Nones

In [27]:
na_count = X.isna().sum()

print(na_count)

Gender                            0
Age                               0
Height                            0
Weight                            0
family_history_with_overweight    0
FAVC                              0
FCVC                              0
NCP                               0
CAEC                              0
SMOKE                             0
CH2O                              0
SCC                               0
FAF                               0
TUE                               0
CALC                              0
MTRANS                            0
dtype: int64


# Encoding Binary and Categorical Features

In [44]:
numeric_feat = X.select_dtypes(include=['int', 'float'])
binary_feat = X[['Gender', 'family_history_with_overweight', 'FAVC', 'SMOKE', 'SCC']]
categorical_features = X[['CAEC', 'CALC', 'MTRANS']]

# binary_features encoding
binary_feat = pd.get_dummies(binary_feat, drop_first=True).astype(int)

# Categorical
print("Categorical features:")
for col in categorical_features.columns:
    print(f'{col}: {np.unique(categorical_features[col])}')

# Ordinal
frequency_map = {'no': 0, 'Sometimes':1, 'Frequently': 2, 'Always': 3}
categorical_features['CAEC'] = categorical_features['CAEC'].map(frequency_map)
categorical_features['CALC'] = categorical_features['CALC'].map(frequency_map)

# Not Ordinal
coded_transport = pd.get_dummies(categorical_features['MTRANS'], drop_first=False).astype(int)
categorical_features.drop("MTRANS", axis=1, inplace=True)
categorical_features = pd.concat([categorical_features, coded_transport], axis=1)

new_X = pd.concat([numeric_feat, categorical_features, binary_feat], axis=1)
new_X

Categorical features:
CAEC: ['Always' 'Frequently' 'Sometimes' 'no']
CALC: ['Frequently' 'Sometimes' 'no']
MTRANS: ['Automobile' 'Bike' 'Motorbike' 'Public_Transportation' 'Walking']


,Age,Height,Weight,FCVC,NCP,CH2O,FAF,TUE,CAEC,CALC,Automobile,Bike,Motorbike,Public_Transportation,Walking,Gender_Male,family_history_with_overweight_yes,FAVC_yes,SMOKE_yes,SCC_yes
0,24.443011,1.699998,81.669950,2.000000,2.983297,2.763573,0.000000,0.976473,1,1,0,0,0,1,0,1,1,1,0,0
1,18.000000,1.560000,57.000000,2.000000,3.000000,2.000000,1.000000,1.000000,2,0,1,0,0,0,0,0,1,1,0,0
2,18.000000,1.711460,50.165754,1.880534,1.411685,1.910378,0.866045,1.673584,1,0,0,0,0,1,0,0,1,1,0,0
3,20.952737,1.710730,131.274851,3.000000,3.000000,1.674061,1.467863,0.780199,1,1,0,0,0,1,0,0,1,1,0,0
4,31.641081,1.914186,93.798055,2.679664,1.971472,1.979848,1.967973,0.931721,1,1,0,0,0,1,0,1,1,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20753,25.137087,1.766626,114.187096,2.919584,3.000000,2.151809,1.330519,0.196680,1,1,0,0,0,1,0,1,1,1,0,0
20754,18.000000,1.710000,50.000000,3.000000,4.000000,1.000000,2.000000,1.000000,2,1,0,0,0,1,0,1,0,1,0,0
20755,20.101026,1.819557,105.580491,2.407817,3.000000,2.000000,1.158040,1.198439,1,0,0,0,0,1,0,1,1,1,0,0
20756,33.852953,1.700000,83.520113,2.671238,1.971472,2.144838,0.000000,0.973834,1,0,1,0,0,0,0,1,1,1,0,0


# Encoding Target

In [46]:
obesity_map = {
    'Insufficient_Weight': 0,
    'Normal_Weight': 1,
    'Overweight_Level_I': 2,
    'Overweight_Level_II': 3,
    'Obesity_Type_I': 4,
    'Obesity_Type_II': 5,
    'Obesity_Type_III': 6
}

y_encoded = y.map(obesity_map)

y_encoded

0        3
1        1
2        0
3        6
4        3
        ..
20753    5
20754    0
20755    5
20756    3
20757    5
Name: NObeyesdad, Length: 20758, dtype: int64

# Final Data and Saving

In [49]:
os.makedirs('Data', exist_ok=True)
data = pd.concat([new_X, y_encoded], axis=1)
print(data.dtypes)

path = os.path.join("Data", "data.csv")
data.to_csv(path)

Age                                   float64
Height                                float64
Weight                                float64
FCVC                                  float64
NCP                                   float64
CH2O                                  float64
FAF                                   float64
TUE                                   float64
CAEC                                    int64
CALC                                    int64
Automobile                              int64
Bike                                    int64
Motorbike                               int64
Public_Transportation                   int64
Walking                                 int64
Gender_Male                             int64
family_history_with_overweight_yes      int64
FAVC_yes                                int64
SMOKE_yes                               int64
SCC_yes                                 int64
NObeyesdad                              int64
dtype: object
